# EfficientSAM reproduction on Google Colab (T4 GPU)

This notebook runs our whole reproduction pipeline for **EfficientSAM (Xiong et al., CVPR 2024)**:

| Step | What it does | Time on a free T4 |
|---|---|---|
| 1 | Setup: GPU check, Google Drive, data, code, packages | ~5 min |
| 2 | Download the released checkpoints (EfficientSAM-Ti/S, MobileSAM, SAM ViT-B/H) | ~5 min |
| 3 | Freeze the evaluation prompts (1,620 objects in 199 held-out images) | ~1 min |
| 4 | **Teacher features**: run SAM ViT-H once over the 800 training images (+ mirrored copies) | ~17 min |
| 5 | **Pretraining**: SAMI (target = SAM ViT-H features) and MAE (target = pixels), 400 epochs each | a few hours |
| 6 | **Fine-tuning**: student from a SAMI, MAE or random start, encoder trained, 100 epochs | ~35 min each |
| 7 | **Ablation**: the same, with the pretrained encoder frozen | faster than step 6 |
| 8 | **Experiment C**: gentler fine-tuning (20 epochs, or encoder LR 1e-5) | ~7 / ~35 min each |
| 8 | **Experiment D**: only 100 or 200 labelled images (same number of training steps) | ~35 min each |
| 8 | **Experiment E**: 5,000 images (data preparation on any runtime, then same number of steps) | ~30 min + ~35 min each |
| 9 | Evaluate every model on the same prompts | a few min each (SAM ViT-H ~10 min) |
| 10 | Summary tables and a zip of all results | ~1 min |

Every training step saves a checkpoint to Google Drive regularly and **resumes automatically**: if Colab
disconnects, run the same cell again. Every step also **skips work that is already finished**, so the notebook can
be re-run from the top at any time.

**Code provenance.** `efficient_sam/` is the official EfficientSAM code (unchanged). Everything in `repro/` was
written by our team, except `build_student_sam()` (adapted from the official `build_efficient_sam()`) and
`sam_loss()` (adapted from our earlier `from_scratch/tinysam` losses). See `repro/README.md`.

**Results** of our runs are in `output/repro_results/` of the repository; this notebook is the record of how they
were produced. Note: each step was run in separate Colab sessions over several days (the free GPU quota runs
out); this clean notebook contains the final version of every cell, so it has no saved outputs.

## 1. Setup
Runtime → Change runtime type → **T4 GPU**. Expected folders on Google Drive:
* `MyDrive/COCO_dataset/` – our COCO subset (`annotations/subset_instances_{train,val}2017.json`, `images/{train,val}2017/`)
* `MyDrive/effsam_work/` – created automatically; holds weights, features, checkpoints and results.

In [ ]:
import os, shutil, subprocess, torch
os.chdir('/content')                                     # never stand inside a folder we are about to replace

# 1. GPU check. When the free quota is used up Colab silently gives a CPU runtime: stop instead of running for hours.
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'No GPU: the free Colab quota is probably used up. Try again later.'

# 2. Google Drive (persistent storage for everything we produce)
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/effsam_work'
for d in ['weights', 'features', 'checkpoints', 'results/gpu_t4']:
    os.makedirs(f'{WORK}/{d}', exist_ok=True)

# 3. Data: copy the COCO subset to the local disk once per session (much faster than reading from Drive)
DATA = '/content/data/COCO_dataset'
if not os.path.exists(DATA):
    shutil.copytree('/content/drive/MyDrive/COCO_dataset', DATA)

# 4. Code: our repository (official efficient_sam/ + our repro/)
BRANCH = 'coco-eda-sharing'                              # branch that holds repro/ (change if it moves)
CODE = '/content/EfficientSAM'
shutil.rmtree(CODE, ignore_errors=True)
subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, 'https://github.com/Mehdi-blip-stack/EfficientSAM', CODE], check=True)

# 5. Packages: SAM (teacher + baselines) and MobileSAM (baseline); torch, numpy, scipy, pycocotools come with Colab
subprocess.run('pip -q install git+https://github.com/facebookresearch/segment-anything.git '
               'git+https://github.com/ChaoningZhang/MobileSAM.git', shell=True, check=True)

# 6. Environment variables read by repro/
os.environ['EFFSAM_REPO'] = CODE
os.environ['COCO_DATA_ROOT'] = DATA
os.environ['REPRO_OUT'] = f'{WORK}/results/gpu_t4'
os.chdir(CODE)
print('ready:', os.getcwd())

## 2. Released checkpoints
EfficientSAM weights come with the repository (`weights/`); SAM and MobileSAM weights are downloaded from their
official sources.

In [ ]:
import zipfile, urllib.request
W = f'{WORK}/weights'
URLS = {'sam_vit_h_4b8939.pth': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
        'sam_vit_b_01ec64.pth': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
        'mobile_sam.pt': 'https://github.com/ChaoningZhang/MobileSAM/raw/master/weights/mobile_sam.pt'}
for name, url in URLS.items():
    if not os.path.exists(f'{W}/{name}'):
        print('downloading', name); urllib.request.urlretrieve(url, f'{W}/{name}')
if not os.path.exists(f'{W}/efficient_sam_vitt.pt'):
    shutil.copy(f'{CODE}/weights/efficient_sam_vitt.pt', W)
if not os.path.exists(f'{W}/efficient_sam_vits.pt'):
    zipfile.ZipFile(f'{CODE}/weights/efficient_sam_vits.pt.zip').extractall(W)
print(sorted(os.listdir(W)))

## 3. Frozen evaluation prompts
For every eligible object of the 200 held-out images (non-crowd, mask ≥ 4 px) we store the ground-truth **box**
and one **click** at the pixel farthest from the mask boundary, plus its size bucket (from the recomputed mask
area) and supercategory. Every model is later tested on exactly these prompts.

In [ ]:
OUT = os.environ['REPRO_OUT']
if not os.path.exists(f'{OUT}/eval_prompts.json'):
    subprocess.run(['python', '-m', 'repro.make_eval_prompts'], check=True)
print('prompts ready')

## 4. Teacher features (SAMI target)
SAM ViT-H looks at each training image and its mirrored copy at 1024×1024. Its 256×64×64 feature map is
average-pooled to the student's 16×16 token grid and cached, together with the 256×256 student inputs.

In [ ]:
FEAT = f'{WORK}/features'
if not os.path.exists(f'{FEAT}/teacher_vit_h_16x16.npy'):
    subprocess.run(['python', '-m', 'repro.teacher_features', '--checkpoint', f'{WORK}/weights/sam_vit_h_4b8939.pth',
                    '--out', FEAT], check=True)
print(sorted(os.listdir(FEAT)))

## 5. Pretraining: SAMI and MAE (400 epochs each)
Same ViT-Tiny encoder, masking (75%), cross-attention decoder and schedule; **only the target differs**:
SAMI reconstructs SAM ViT-H features (MSE over all tokens, paper Eq. 1), MAE reconstructs normalised pixels
(MSE over masked tokens). Output: `checkpoints/{sami,mae}/encoder_final.pt`.

In [ ]:
for target in ['sami', 'mae']:
    out = f'{WORK}/checkpoints/{target}'
    if os.path.exists(f'{out}/encoder_final.pt'):
        print('already done:', target); continue
    print('=====', target, flush=True)
    subprocess.run(['python', '-m', 'repro.pretrain', '--target', target, '--features', FEAT, '--out', out], check=True)

## 6–8. Fine-tuning runs
All runs fine-tune the EfficientSAM-Ti architecture at 256 px on the 800 training images (5,827 masks) with
box or click prompts and SAM's loss (20 × focal + Dice on the best of 3 masks, + IoU MSE). They differ only in:

| Run | Encoder start | Setting | Purpose |
|---|---|---|---|
| `ft_sami`, `ft_mae`, `ft_random` | SAMI / MAE / random | encoder trained, 100 epochs, encoder LR 1e-4 | Reproduction (report Sec. 6) |
| `ft_*_frozen` | SAMI / MAE / random | encoder frozen (neck + decoder trained) | Ablation (Sec. 7.2) |
| `ft_*_short` | SAMI / MAE / random | 20 epochs | Experiment C |
| `ft_*_lowlr` | SAMI / MAE / random | encoder LR 1e-5 | Experiment C |
| `ft_*_n100`, `ft_*_n200` | SAMI / MAE / random | masks of only the first 100 / 200 images, 800 / 400 epochs | Experiment D (Sec. 7.4) |
| `ft_*_5k` | SAMI / MAE / random | 5,000 images (our 800 + 4,200 random COCO train2017), 16 epochs | Experiment E (Sec. 7.5) |

Every setting uses the same number of training steps as the 100-epoch, 800-image runs, except the 20-epoch runs
of Experiment C. Experiment E first needs its data: run the next cell once (it needs **no GPU**, so a CPU runtime
is fine); it downloads 4,200 extra COCO images and prepares them like the original 800.

In [ ]:
FEAT5K = f'{WORK}/features_5k'
if os.path.exists(f'{FEAT5K}/prepare_5k_info.json'):
    print('already prepared:', open(f'{FEAT5K}/prepare_5k_info.json').read())
else:
    subprocess.run(['python', '-m', 'repro.prepare_5k', '--base_data', DATA, '--work_data', '/content/data/COCO_5k',
                    '--out_features', '/content/features_5k', '--n', '5000'], check=True)
    shutil.copytree('/content/features_5k', FEAT5K + '_copying')
    os.rename(FEAT5K + '_copying', FEAT5K)            # appears under its real name only once the copy is complete
    print('done:', open(f'{FEAT5K}/prepare_5k_info.json').read())

In [ ]:
ENC = {'sami': f'{WORK}/checkpoints/sami/encoder_final.pt', 'mae': f'{WORK}/checkpoints/mae/encoder_final.pt', 'random': None}
SETTINGS = {'': [], '_frozen': ['--freeze_encoder'], '_short': ['--epochs', '20'], '_lowlr': ['--lr_encoder', '1e-5'],
            '_n100': ['--limit', '100', '--epochs', '800', '--warmup', '40', '--save_every', '20'],
            '_n200': ['--limit', '200', '--epochs', '400', '--warmup', '20', '--save_every', '10'],
            '_5k': ['--epochs', '16', '--warmup', '1']}
RUN_SETTINGS = ['', '_frozen', '_short', '_lowlr', '_n100', '_n200', '_5k']   # remove entries you do not want to run
FEAT5K = f'{WORK}/features_5k'

for suffix in RUN_SETTINGS:
    for init in ['sami', 'mae', 'random']:
        name = f'ft_{init}{suffix}'
        out = f'{WORK}/checkpoints/{name}'
        if os.path.exists(f'{out}/model_final.pt'):
            print('already done:', name); continue
        feats = FEAT
        if suffix == '_5k':
            if not os.path.exists(f'{FEAT5K}/prepare_5k_info.json'):
                print('Experiment E data not prepared yet, skipping:', name); continue
            feats = '/content/features_5k'                    # local copy: much faster to read than Drive
            if not os.path.exists(f'{feats}/finetune_targets_256.npz'):
                shutil.copytree(FEAT5K, feats, dirs_exist_ok=True)
        cmd = ['python', '-m', 'repro.finetune', '--init', init, '--features', feats, '--out', out] + SETTINGS[suffix]
        if ENC[init]:
            cmd += ['--encoder', ENC[init]]
        print('=====', name, flush=True)
        subprocess.run(cmd, check=True)

## 9. Evaluation
Each model runs in its own process (so peak memory is measured per model) on the frozen prompts. For every
prompt the model returns 3 masks; we score the one with the highest predicted IoU and also record the
best-of-3 (oracle). Output per model: `per_instance_<model>.csv` and `efficiency_<model>.json`.

In [ ]:
W = f'{WORK}/weights'
MODELS = {'effsam_ti': f'{W}/efficient_sam_vitt.pt', 'effsam_s': f'{W}/efficient_sam_vits.pt',
          'mobile_sam': f'{W}/mobile_sam.pt', 'sam_vit_b': f'{W}/sam_vit_b_01ec64.pth', 'sam_vit_h': f'{W}/sam_vit_h_4b8939.pth'}
for suffix in RUN_SETTINGS:
    for init in ['sami', 'mae', 'random']:
        MODELS[f'ours_{init}{suffix}'] = f'{WORK}/checkpoints/ft_{init}{suffix}/model_final.pt'

for model, weights in MODELS.items():
    if os.path.exists(f'{OUT}/per_instance_{model}.csv'):
        print('already done:', model); continue
    if not os.path.exists(weights):
        print('no weights yet, skipping:', model); continue
    print('=====', model, flush=True)
    subprocess.run(['python', '-m', 'repro.evaluate_model', '--model', model, '--weights', weights], check=True)

## 10. Summary tables and results zip
Writes `results_summary.md` and `results_tables.json` (overall, by size, by supercategory, category-macro, paired
differences with 95% bootstrap CIs over images, efficiency) and zips the whole results folder.

In [ ]:
subprocess.run(['python', '-m', 'repro.summarize_results'], check=True)
shutil.make_archive(f'{WORK}/results_gpu_t4', 'zip', OUT)
print(open(f'{OUT}/results_summary.md').read()[:3000])